## Csomagok betöltése 

In [77]:
import copy
import math
import random
import numpy as np
import torch
import torch.nn as nn
from tensorflow.python.framework.random_seed import set_seed
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models
import medmnist
from medmnist import INFO
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix
import os
import json
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## Random seed értékek beállítása

**Később használva lesznek a backbone és a single-task tanításnál**

In [78]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Használt eszköz: {device}")


Használt eszköz: cuda


## Train - Validation - Test adathalmazok szétválasztása a hivatalos benchmark szerint

In [79]:
def get_dataloaders(dataset_name, batch_size=128, max_train_samples=None, max_eval_samples=None):
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info['python_class'])

    data_transform = transforms.Compose([ # Normálja és skálázza az adatokat
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    # A hivatalos MNIST splitek vannak stratidy szempontból jók
    train_dataset = DataClass(split='train', transform=data_transform, download=True, as_rgb=True)
    val_dataset   = DataClass(split='val',   transform=data_transform, download=True, as_rgb=True)
    test_dataset  = DataClass(split='test',  transform=data_transform, download=True, as_rgb=True)

    # Shuffle keveri hogy ne tanuljon rá a mintára
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,drop_last=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,drop_last=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,drop_last=True)


    return train_loader, val_loader, test_loader, info

## TxT performance mátrix vizualizációja és mentése

A kísérlet során a modell szekvenciálisan, egymás után tanulja meg a kijelölt $T$ darab feladatot egy előre meghatározott sorrendben:
$$\mathcal{T} = \{\tau_1, \tau_2, \dots, \tau_T\}$$

Minden egyes $i$-edik feladat ($\tau_i$) betanításának befejezése után a modell azonnali kiértékelésre kerül az összes feladat ($\tau_j$, ahol $j \in \{1, \dots, T\}$) független tesztkészletén.

A kiértékelés eredményét az $R \in \mathbb{R}^{T \times T}$ mátrix rögzíti, ahol:
* **Sorindex ($i \in \{1, \dots, T\}$):** A tanítási fázis, azaz az utoljára betanított feladat sorszáma.
* **Oszlopindex ($j \in \{1, \dots, T\}$):** A tesztelt feladat azonosítója.
* ***$R_{i,j}$:*** A modell tesztpontossága a $\tau_j$ feladaton, közvetlenül a $\tau_i$ feladat tanulásának lezárultakor.

$$R = \begin{bmatrix}
R_{1,1} & R_{1,2} & \dots & R_{1,T} \\
R_{2,1} & R_{2,2} & \dots & R_{2,T} \\
\vdots  & \vdots  & \ddots & \vdots  \\
R_{T,1} & R_{T,2} & \dots & R_{T,T}
\end{bmatrix}$$

* **Főátló ($R_{j,j}$):** Az egyes feladatok önálló betanulásakor elért **azonnali pontossága**.

* **Főátló alatti elemek ($i > j$):** A korábban megtanult feladatok ($\tau_j$) későbbi teljesítménye az újabb feladatok ($\tau_i$) elsajátítása után. Az oszlopok lefelé történő mozgása írja le a **catastrophic forgetting** dinamikáját.

* **Főátló feletti elemek ($i < j$):** A modell **előzetes teljesítménye** olyan feladatokon, amelyeket a tanítási folyamatban formálisan még nem látott.

* **Utolsó sor ($R_{T,j}$):** A teljes tanítási folyamat végén megmaradt végső modell tudásállapota az összes feladaton.

In [80]:
RESULT_DIR = os.path.abspath("results")
os.makedirs(RESULT_DIR, exist_ok=True)


def save_publication_matrix(
    R_matrix, task_order, experiment_name="sequential_baseline"
):
    plt.close("all")
    plt.rcParams.update(plt.rcParamsDefault)
    sns.set_theme(style="white")

    row_labels = [f"Step {i+1} ({t})" for i, t in enumerate(task_order)]
    col_labels = task_order
    df_matrix = pd.DataFrame(R_matrix, index=row_labels, columns=col_labels)

    # Főátló feletti értékek elfedése
    mask = np.triu(np.ones_like(R_matrix, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(8.5, 6.8), dpi=300)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    heatmap = sns.heatmap(
        df_matrix,
        mask=None,
        annot=True,
        fmt=".4f", # Hány tizedesjegy pontosság legyen
        annot_kws={"size": 11, "weight": "bold"},
        cmap="Blues",
        vmin=0.0,
        vmax=100.0,
        linewidths=1.2,
        linecolor="white",
        cbar_kws={
            "label": "Test Accuracy (%)",
            "shrink": 0.88,
            "pad": 0.04,
            "ticks": np.arange(0, 101, 10),
        },
        ax=ax,
    )

    # Szövegszín korrekció: sötét kék háttéren fehér, világoson fekete
    for text in heatmap.texts:
        try:
            val = float(text.get_text())
            text.set_color("black" if val < 50.0 else "white")
        except ValueError:
            pass

    ax.set_title(
        f"T x T Performance Matrix - {experiment_name}",
        fontsize=13,
        fontweight="bold",
        color="black",
        pad=18,
    )
    ax.set_xlabel(
        "Evaluated Task (Test Set)",
        fontsize=11,
        fontweight="semibold",
        color="black",
        labelpad=12,
    )
    ax.set_ylabel(
        "Training Phase (After Task)",
        fontsize=11,
        fontweight="semibold",
        color="black",
        labelpad=12,
    )

    ax.tick_params(colors="black", labelsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", color="black")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, color="black")

    cbar = heatmap.collections[0].colorbar
    cbar.ax.tick_params(color="black", labelcolor="black", labelsize=10)
    cbar.set_label(
        "Test Accuracy (%)",
        fontsize=10.5,
        fontweight="semibold",
        color="black",
        labelpad=10,
    )

    plt.tight_layout()

    # Mentés biztonságos abszolút útvonalra és tiszta paraméterezéssel
    img_path = os.path.join(RESULT_DIR, f"{experiment_name}_heatmap.png")
    fig.savefig(img_path, dpi=300, facecolor="white", bbox_inches="tight")
    # plt.show()

    print(f"Ábra sikeresen elmentve: {img_path}")

## Javasolt mérőszámok kiszámítása és mentése JSON fájlba

### 1. Végső Átlagos Pontosság (`final_average_accuracy` – $A_T$)
A teljes tanulási folyamat legvégén (a $T$-edik feladat után) a modell tesztkészleteken mért pontosságainak számtani átlaga. Megadja a modell globális, végső megtartott tudásszintjét:
$$A_T = \frac{1}{T} \sum_{j=1}^{T} R_{T,j}$$
* **Értelmezés:** A **continous learning-ben** a cél ennek maximalizálása. Sequential fine-tuning esetén a korábbi feladatok felejtése miatt ez az érték drasztikusan elmarad a Joint-training (felső referencia) szintjétől.

---

### 2. Lépésenkénti Átlagos Pontosság (`step_average_accuracies` – $A_i$)
Az $i$-edik feladat megtanulása után az **addig látott feladatok** ($j \in \{1, \dots, i\}$) átlagos pontossága:
$$A_i = \frac{1}{i} \sum_{j=1}^{i} R_{i,j} \quad (i = 1, \dots, T)$$
* **Értelmezés:** Nyomon követi a rendszer átlagos tudásának trajektóriáját az idővonal mentén. Megmutatja, hogy egy új feladat belépése mennyire veti vissza az addig felépített átlagos teljesítményt.

---

### 3. Visszamenőleges Transzfer (`backward_transfer` – $\text{BWT}$)
Számszerűsíti, hogy a későbbi feladatok ($\tau_{j+1}, \dots, \tau_T$) elsajátítása hogyan befolyásolta a korábbi feladatok ($\tau_j$) megtartott tudását a saját tanításukkor elért **csúcs pontossághoz** ($R_{j,j}$) viszonyítva:
$$\text{BWT} = \frac{1}{T-1} \sum_{j=1}^{T-1} \left( R_{T,j} - R_{j,j} \right)$$
* **Feladatonkénti BWT (`bwt_per_task`):**
  $$\text{BWT}_j = R_{T,j} - R_{j,j} \quad (j = 1, \dots, T-1)$$
* **Értelmezés:** 
  * $\text{BWT} < 0$: **Catastrophic forgetting**. Az új feladatok tanulása rontotta a korábbi feladatokon elért pontosságot. Minél nagyobb a negatív érték annál jobban felejtett a modell
  * $\text{BWT} = 0$: Tökéletes megőrizte a korábbi tudását, de az új feladatok nem segítették a régi feladatok teljesítményét. 
  * $\text{BWT} > 0$: **Pozitív visszamenőleges transzfer**. Az újabb feladatok tanulása során elsajátított közös reprezentációk még javították is a korábbi feladatok általánosítását.

---

### 4. Átlagos Felejtés (`average_forgetting` – $F$)


Minden korábbi feladatra vizsgálja a tanítás során elért maximális teljesítmény és a legvégső lépésben ($T$) maradt pontosság közötti veszteséget, pozitív mérőszámként definiálva:
$$F = \frac{1}{T-1} \sum_{j=1}^{T-1} f_j$$
ahol a $j$-edik feladat felejtése (`forgetting_per_task`):
$$f_j = \max_{k \in \{j, \dots, T-1\}} R_{k,j} - R_{T,j}$$
* **BWT vs Average Forgetting:** Míg a BWT azt vizsgálja, hogy a folyamat legvégén hogyan áll a múltbeli tudás az eredetihez képest, addig az Average Forgetting azt számszerűsíti, hogy a feladatok a tanulási folyamat során maximális teljesítményükhöz képest átlagosan mennyit romlottak.


---

### 5. Előrefelé Transzfer – Tanítás Előtt (`forward_transfer_pre` – $\text{FWT}_{\text{pre}}$)
Azt vizsgálja, hogy a modell hogyan teljesít egy adott feladaton **közvetlenül annak betanítása előtt**, kizárólag a korábban látott feladatokból származó tudás (backbone) és még a véletlenszerű kimeneti fej alapján: 
$$\text{FWT}_{\text{pre}} = \frac{1}{T-1} \sum_{j=2}^{T} R_{j-1, j}$$
* **Feladatonkénti tanítás előtti transzfer (`fwt_pre_per_task`):**
  $$\text{FWT}_{\text{pre}, j} = R_{j-1, j} \quad (j = 2, \dots, T)$$
* **Értelmezés:** A mátrix főátló feletti közvetlen szomszédos elemeinek átlaga. Multi-head architektúrában a $j$-edik feladat kimeneti feje ebben a fázisban még véletlenszerű inicializálású, így ez az érték a még nem kalibrált fej és a korábbi feladatokon edzett reprezentációk együttes zéró-shot bázisát mutatja.

---

### 6. Előrefelé Transzfer – Single-Task Referenciával (`forward_transfer_single` – $\text{FWT}_{\text{single}}$)
Azt vizsgálja, hogy a folyamatos tanulás során az előzetesen megszerzett reprezentációk segítik-e a $j$-edik feladat jobb megtanulását ahhoz képest, mintha azt teljesen izoláltan, nulláról (Single-task Oracle) tanultuk volna meg:
$$\text{FWT}_{\text{single}} = \frac{1}{T-1} \sum_{j=2}^{T} \left( R_{j,j} - b_{\text{single}, j} \right)$$
* **Feladatonkénti FWT (`fwt_single_per_task`):**
  $$\text{FWT}_{\text{single}, j} = R_{j,j} - b_{\text{single}, j} \quad (j = 2, \dots, T)$$
  ahol $b_{\text{single}, j}$ a $\tau_j$ feladaton izoláltan, önállóan betanított modell tesztpontossága.
* **Értelmezés:**
  * $\text{FWT}_{\text{single}} > 0$: **Pozitív előrehatás**. A korábbi feladatok tudásbázisa előnyt biztosított az új feladat elsajátításakor.
  * $\text{FWT}_{\text{single}} < 0$: **Negatív interferencia / plaszticitásvesztés**. A korábbi feladatokhoz merevedett súlyok gátolták az új feladatra való optimális ráállást.
  * Ha a referenciavektor ($b_{\text{single}}$) nincs megadva, az értéke `null`, a feladatonkénti szótár pedig `{}`.

In [81]:
RESULT_DIR = os.path.abspath("results")
os.makedirs(RESULT_DIR, exist_ok=True)

def compute_and_save_cl_metrics(R_matrix, task_order, seed=42, experiment_name="sequential_baseline", b_single_dict=None):
    """
    A T x T mátrix vizualizációjától teljesen függetlenül:
    1. Kiszámítja az összes kötelező CL metrikát (Final Acc, Step Acc, BWT, Forgetting, FWT).
    2. Elmenti az eredményeket strukturált JSON és összefoglaló CSV formátumban.
    3. Kiír egy áttekinthető szöveges összegzést a konzolra.
    """
    T = len(task_order)
    R = np.array(R_matrix, dtype=float)

    # 1. Final Average Accuracy (A_T) - utolsó sor átlaga
    final_avg_acc = float(np.mean(R[T-1, :]))

    # 2. Lépésenkénti Average Accuracy (A_i) - i-edik lépés után az addigi taskok átlaga
    step_avg_acc = {}
    for i in range(T):
        step_avg_acc[f"Step_{i+1}_{task_order[i]}"] = round(float(np.mean(R[i, :i+1])), 4)

    # 3. Backward Transfer (BWT) - utolsó sor vs. főátló az első T-1 taskra
    bwt_per_task = {}
    for j in range(T - 1):
        diff = R[T-1, j] - R[j, j]
        bwt_per_task[task_order[j]] = round(float(diff), 4)
    bwt = float(np.mean(list(bwt_per_task.values()))) if (T > 1) else 0.0

    # 4. Average Forgetting (F) - csúcsteljesítmény vs. utolsó sor
    forgetting_per_task = {}
    for j in range(T - 1):
        peak = np.max(R[j:T-1, j])
        f_j = peak - R[T-1, j]
        forgetting_per_task[task_order[j]] = round(float(f_j), 4)
    avg_forgetting = float(np.mean(list(forgetting_per_task.values()))) if (T > 1) else 0.0

    # 5. Forward Transfer (FWT)
    # A) Tanítás előtti (zero-shot) FWT a főátló feletti közvetlen szomszédokból
    fwt_pre_per_task = {}
    for j in range(1, T):
        fwt_pre_per_task[task_order[j]] = round(float(R[j-1, j]), 4)
    fwt_pre = float(np.mean(list(fwt_pre_per_task.values()))) if (T > 1) else 0.0

    # B) Single-task referencia alapú FWT (ha rendelkezésre áll)
    fwt_single = None
    fwt_single_per_task = {}
    if b_single_dict is not None:
        diffs = []
        for j in range(1, T):
            t_name = task_order[j]
            if t_name in b_single_dict:
                d = R[j, j] - b_single_dict[t_name]
                fwt_single_per_task[t_name] = round(float(d), 4)
                diffs.append(d)
        if len(diffs) > 0:
            fwt_single = round(float(np.mean(diffs)), 4)

    # Összesített dictionary mentéshez
    metrics_summary = {
        "experiment_name": experiment_name,
        "seed": seed,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "task_order": task_order,
        "final_average_accuracy": round(final_avg_acc, 4),
        "backward_transfer": round(bwt, 4),
        "average_forgetting": round(avg_forgetting, 4),
        "forward_transfer_pre": round(fwt_pre, 4),
        "forward_transfer_single": fwt_single,
        "step_average_accuracies": step_avg_acc,
        "bwt_per_task": bwt_per_task,
        "forgetting_per_task": forgetting_per_task,
        "fwt_pre_per_task": fwt_pre_per_task,
        "fwt_single_per_task": fwt_single_per_task
    }

    # 1. Mentés JSON fájlba (teljes strukturált riport)
    json_path = os.path.join(RESULT_DIR, f"{experiment_name}_seed{seed}_metrics.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(metrics_summary, f, indent=4, ensure_ascii=False)

    # 3. Áttekinthető szöveges kiírás a konzolra
    # print("\n" + "=" * 55)
    # print(f" CONTINUAL LEARNING METRIKÁK (Seed: {seed})")
    # print("=" * 55)
    # print(f" Final Average Accuracy (A_T): {final_avg_acc:.2f}%")
    # print(f" Backward Transfer (BWT):      {bwt:.2f}%")
    # print(f" Average Forgetting (F):       {avg_forgetting:.2f}%")
    # print(f" Forward Transfer (Zero-shot): {fwt_pre:.2f}%")
    # if fwt_single is not None:
    #     print(f" Forward Transfer (Single-ref):{fwt_single:.2f}%")
    # print("-" * 55)
    # print("Lépésenkénti átlagos pontosságok (A_i):")
    # for step, acc in step_avg_acc.items():
    #     print(f"  {step}: {acc:.2f}%")
    # print("-" * 55)
    # print("Felejtés feladatonként (Forgetting):")
    # for t_name, f_val in forgetting_per_task.items():
    #     print(f"  {t_name}: {f_val:.2f}%")
    # print("=" * 55)
    print(f"Eredmények elmentve:")
    print(f"  -> JSON: {json_path}")

    return metrics_summary

Res-Net-18 mert ez van a MedMNIST hivatlos kutatási keretrendszerében

In [82]:
def create_model(num_classes):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


In [83]:
class MultiHeadResNet18(nn.Module):
    def __init__(self, task_classes_dict):
        """
        task_classes_dict: Egy szótár, ami megadja, hogy melyik taskhoz hány osztály tartozik.
                           Pl. {'pathmnist': 9, 'dermamnist': 7, 'octmnist': 4, 'pneumoniamnist': 2, 'organamnist': 11}
        """
        super(MultiHeadResNet18, self).__init__()

        # 1. Közös 2D Backbone (ResNet-18, weights=None a tiszta FWT referenciához)
        base_resnet = models.resnet18(weights=None)

        # Levesszük az eredeti fc réteget, hogy csak a konvolúciós feature extractor maradjon
        self.backbone = nn.Sequential(*list(base_resnet.children())[:-1])
        in_features = base_resnet.fc.in_features # általában 512

        # 2. Task-specifikus kimeneti fejek (Task-specific heads) nn.ModuleDict-ben
        self.heads = nn.ModuleDict()
        for task_name, num_classes in task_classes_dict.items():
            self.heads[task_name] = nn.Linear(in_features, num_classes)

    def forward(self, x, task_name):
        # 1. Átküldjük a képet a közös backbone-on
        features = self.backbone(x)
        features = torch.flatten(features, 1) # [Batch_size, 512]

        # 2. Kiválasztjuk a task_name (task_id) alapján a megfelelő fejet
        if task_name not in self.heads:
            raise ValueError(f"Ismeretlen task név: {task_name}")

        out = self.heads[task_name](features)
        return out

Kiértékeli a modell teljesítményét

In [84]:
def evaluate_detailed(model, loader, task_type, criterion, eval_task=None):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            # outputs = model(images,task_name=eval_task)

            if hasattr(model, 'heads') and eval_task is not None:
                outputs = model(images, task_name=eval_task) # Itt adom át az éppen futtatott modell nevét ahhoz hogy a backboneból a megfelelő fejet találja meg
            else:
                outputs = model(images)

            if task_type == 'multi-label, binary-class':
                labels = labels.to(device).float()
                loss = criterion(outputs, labels)
                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).float()
            else:
                labels = labels.squeeze(-1).long().to(device)
                loss = criterion(outputs, labels)
                probs = F.softmax(outputs, dim=1)
                preds = outputs.argmax(dim=1)

            total_loss += loss.item() * images.size(0)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)

    # Biztonságos metrikaszámítás feladattípus szerint
    if task_type == 'multi-label, binary-class':
        accuracy = (all_preds == all_labels).mean() * 100.0
        precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
        try:
            roc_auc = roc_auc_score(all_labels, all_probs, average='macro')
        except ValueError:
            roc_auc = 0.0
        cm = None  # Multi-label esetén a konfúziós mátrix nem egyértelmű mátrixként adódik
    else:
        accuracy = (all_preds == all_labels).mean() * 100.0
        precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
        cm = confusion_matrix(all_labels, all_preds)
        try:
            n_classes = all_probs.shape[1]
            y_true_onehot = np.eye(n_classes)[all_labels]
            roc_auc = roc_auc_score(y_true_onehot, all_probs, multi_class='ovr', average='macro')
        except ValueError:
            roc_auc = 0.0

    metrics_dict = {
        "loss": round(avg_loss,4),
        "accuracy": float(round(accuracy,4)),
        "precision": round(precision * 100.0,4),
        "recall": round(recall * 100.0,4),
        "f1_score": round(f1 * 100.0,4),
        "roc_auc": round(roc_auc,4)
    }

    return metrics_dict

Egyszeri folyamatos tanulás

In [85]:
def train_single_task(dataset_name, epochs=3, batch_size=128, lr=1e-3, max_train_samples=2000, max_eval_samples=500):
    seed=set_seed(42)
    print(f"\n==========================================")
    print(f"Adathalmaz: {dataset_name.upper()}")
    train_loader, val_loader, test_loader, info = get_dataloaders(
        dataset_name, batch_size, max_train_samples, max_eval_samples
    )

    task_type = info['task']
    num_classes = len(info['label'])
    print(f"Feladat: {task_type} | Osztályok száma: {num_classes}")
    print(f"Használt mintaszám -> Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")

    model = create_model(num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss() if task_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            optimizer.zero_grad()
            outputs = model(images)

            labels = labels.to(device).float() if task_type == 'multi-label, binary-class' else labels.squeeze(-1).long().to(device)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)
        val_metrics = evaluate_detailed(model, val_loader, task_type, criterion)
        val_loss=val_metrics["loss"]
        val_acc=val_metrics["accuracy"]
        # display(val_metrics)
        print(f"Epoch [{epoch}/{epochs}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    test_metrics = evaluate_detailed(model, test_loader, task_type, criterion)
    test_loss=test_metrics["loss"]
    test_acc=test_metrics["accuracy"]
    # display(test_metrics)
    # print(f"--> VÉGSŐ TESZT PONTOSSÁG ({dataset_name.upper()}): {test_acc:.2f}%")
    # save_results_to_file(test_metrics)
    return model, test_acc

In [86]:
# 1. PathMNIST (~90 000 train minta)
model_path_full, acc_path_full = train_single_task('pathmnist', epochs=1, max_train_samples=None, max_eval_samples=None)

# 2. DermaMNIST (~7 000 train minta)
model_derma, acc_derma = train_single_task('dermamnist', epochs=1, max_train_samples=None, max_eval_samples=None)

# 3. OCTMNIST (~97 000 train minta)
model_oct, acc_oct = train_single_task('octmnist', epochs=1, max_train_samples=None, max_eval_samples=None)

# 4. PneumoniaMNIST (~4 700 train minta)
model_pneumonia, acc_pneumonia = train_single_task('pneumoniamnist', epochs=1, max_train_samples=None,
                                                   max_eval_samples=None)

# 5. Organamnist (~34561 train minta)
model_organ, acc_organ = train_single_task('organamnist', epochs=1, max_train_samples=None, max_eval_samples=None)

single_task_dict = {
    'pathmnist': acc_path_full,
    'dermamnist': acc_derma,
    'octmnist': acc_oct,
    'pneumoniamnist': acc_pneumonia,
    'organamnist': acc_organ,
}

# (Opcionális, de ajánlott) Mentsd el JSON-be, hogy ne kelljen újra futtatni:
with open("results/single_task_reference.json", "w") as f:
    json.dump(single_task_dict, f, indent=4)



Adathalmaz: PATHMNIST
Feladat: multi-class | Osztályok száma: 9
Használt mintaszám -> Train: 89996, Val: 10004, Test: 7180
Epoch [1/1] - Train Loss: 0.6731 | Val Loss: 0.8850 | Val Acc: 66.26%

Adathalmaz: DERMAMNIST
Feladat: multi-class | Osztályok száma: 7
Használt mintaszám -> Train: 7007, Val: 1003, Test: 2005
Epoch [1/1] - Train Loss: 0.9400 | Val Loss: 0.7201 | Val Acc: 70.09%

Adathalmaz: OCTMNIST
Feladat: multi-class | Osztályok száma: 4
Használt mintaszám -> Train: 97477, Val: 10832, Test: 1000
Epoch [1/1] - Train Loss: 0.5407 | Val Loss: 0.4066 | Val Acc: 85.61%

Adathalmaz: PNEUMONIAMNIST
Feladat: binary-class | Osztályok száma: 2
Használt mintaszám -> Train: 4708, Val: 524, Test: 624
Epoch [1/1] - Train Loss: 0.1968 | Val Loss: 0.5091 | Val Acc: 86.13%

Adathalmaz: ORGANAMNIST
Feladat: multi-class | Osztályok száma: 11
Használt mintaszám -> Train: 34561, Val: 6491, Test: 17778
Epoch [1/1] - Train Loss: 0.2906 | Val Loss: 0.1704 | Val Acc: 96.19%


Tanítás


In [ ]:
CURRENT_SEED=set_seed(42)
# 1. Előre definiált task sorrend és osztályszámaik
task_order = ['pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist', 'organamnist']
task_classes = {
    'pathmnist': 9,
    'dermamnist': 7,
    'octmnist': 4,
    'pneumoniamnist': 2,
    'organamnist': 11
}

# 2. Példányosítjuk a Multi-head modellt
model = MultiHeadResNet18(task_classes).to(device)

# 3. T x T Performance Mátrix inicializálása
T = len(task_order)
R_matrix = np.zeros((T, T))

# 4. Külső ciklus: Végigmegyünk a feladatokon a megadott sorrendben
for i, train_task in enumerate(task_order):
    print(f"\n==========================================")
    print(f"Fázis [{i+1}/{T}]: Tanítás a(z) {train_task.upper()} adathalmazon")
    print(f"==========================================s")

    # Letöltjük az aktuális task adagolóit (train, val, test)
    train_loader, val_loader, test_loader, info = get_dataloaders(train_task, batch_size=128)
    task_type = info['task']
    criterion = nn.BCEWithLogitsLoss() if task_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()

    # Optimizer (csak a modell paramétereit frissítjük)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Egyszerű tanító ciklus (early stopping a validation szetten)
    best_val_acc = 0.0
    epochs = 1 # Teszthez 5, éles futáshoz 15-20 epoch

    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            optimizer.zero_grad()

            # Forward a megfelelő task_name átadásával a Multi-head modellnek
            outputs = model(images, task_name=train_task)

            if task_type == 'multi-label, binary-class':
                labels = labels.to(device).float()
            else:
                labels = labels.squeeze(-1).long().to(device)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation ellenőrzés
        val_metrics = evaluate_detailed(model, val_loader, task_type, criterion, eval_task=train_task)
        val_acc=val_metrics["accuracy"]
        print(f"Epoch {epoch}/{epochs} - Val Acc ({train_task}): {val_acc:.2f}%")

    # --- 5. KIÉRTÉKELÉS AZ ÖSSZES EDDIG LÁTOTT TASKON (Mátrix kitöltése) ---
    # print(f"\n-> Kiértékelés az összes taskokon a(z) {train_task} tanítása után:")
    model.eval()
    with torch.no_grad():
        for j,eval_task in enumerate(task_order): # Csak az eddig látott taskokat (vagy az összeset FWT-hez) teszteljük
            # Betöltjük az eval_task teszt loaderét
            _, _, eval_test_loader, eval_info = get_dataloaders(eval_task, batch_size=128)
            eval_type = eval_info['task']
            eval_crit = nn.BCEWithLogitsLoss() if eval_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()

            test_metrics = evaluate_detailed(model, eval_test_loader, eval_type, eval_crit, eval_task=eval_task)
            test_acc=test_metrics["accuracy"]

            R_matrix[i, j] = test_acc
            # print(f"   Test Acc on {eval_task}: {test_acc:.2f}%")

print("\n=== VÉGSŐ T x T PERFORMANCE MÁTRIX ===")
save_publication_matrix(R_matrix=R_matrix,task_order=task_order, experiment_name=f"sequential_baseline{CURRENT_SEED}")


# --- 2. AZ ÚJ METRIKÁK KISZÁMÍTÁSA ÉS KÜLÖN KEZELÉSE ---
metrics = compute_and_save_cl_metrics(
    R_matrix=R_matrix,
    task_order=task_order,
    seed=CURRENT_SEED,
    experiment_name="sequential_baseline",
    b_single_dict=single_task_dict # Ha később lesz single-task referenciád, ide átadható
)

## 2. Munkacsomag: Előzetes Task Hasonlóság (Similarity) Mérése

In [ ]:
def compute_task_similarity_matrix(task_list, num_samples=500):
    """
    Kiszámítja az adathalmazok közötti távolságot egy előtanított ResNet-18
    feature centroidjainak koszinusz-távolsága alapján (nem használ tesztadatot).
    """
    print("--- Előzetes Task Similarity elemzés futtatása ---")
    feature_extractor = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
    feature_extractor.fc = nn.Identity()
    feature_extractor.eval()

    centroids = {}
    with torch.no_grad():
        for task_name in task_list:
            train_loader, _, _, _ = get_dataloaders(task_name, batch_size=64, max_train_samples=num_samples)
            features_list = []
            for images, _ in train_loader:
                images = images.to(device)
                feats = feature_extractor(images)
                features_list.append(feats.cpu())
            all_feats = torch.cat(features_list, dim=0)
            centroid = torch.mean(all_feats, dim=0)
            centroid = centroid / torch.norm(centroid, p=2) # L2 normalizálás
            centroids[task_name] = centroid

    num_tasks = len(task_list)
    dist_matrix = np.zeros((num_tasks, num_tasks))
    for i in range(num_tasks):
        for j in range(num_tasks):
            t1, t2 = task_list[i], task_list[j]
            # Koszinusz távolság: 1 - cosine_similarity
            cos_sim = torch.dot(centroids[t1], centroids[t2]).item()
            dist_matrix[i, j] = abs(round(1.0 - cos_sim, 4))

    df_dist = pd.DataFrame(dist_matrix, index=task_list, columns=task_list)
    df_dist.to_csv("results/task_distance_matrix.csv")

    plt.close("all")
    plt.rcParams.update(plt.rcParamsDefault)
    sns.set_theme(style="white")

    # Hasonlósági hőtérkép rajzolása
    fig, ax = plt.subplots(figsize=(6.5, 5.2), dpi=300)
    sns.heatmap(df_dist, annot=True, fmt=".3f", cmap="Blues_r", ax=ax, cbar_kws={'label': 'Koszinusz távolság (kisebb = hasonlóbb)'})
    ax.set_title("Előzetes Task Távolság Mátrix (Feature Cosine Distance)", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/task_similarity_heatmap.png", dpi=300)
    plt.show()

    print("Task távolságok elmentve: results/task_distance_matrix.csv és results/task_similarity_heatmap.png")
    return df_dist

# Futtatás az 5 adathalmazra
task_pool = ['pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist', 'organamnist']
dist_df = compute_task_similarity_matrix(task_pool, num_samples=500)

## 2. Munkacsomag: Task Order Kísérletek (4 sorrend összehasonlítása)

In [98]:
# 1. Kísérleti sorrendek definíciója
def generate_random_order(task_pool, seed):
    rng = random.Random(seed)
    return rng.sample(task_pool, len(task_pool))

base_tasks = ['pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist', 'organamnist']

experiment_orders = {
    "similar_first": ['dermamnist', 'pathmnist', 'organamnist', 'pneumoniamnist', 'octmnist'],

    "dissimilar_alternating": ['dermamnist', 'octmnist', 'pathmnist', 'pneumoniamnist', 'organamnist'],

    "random_order_1": generate_random_order(base_tasks, seed=101),

    "random_order_2": generate_random_order(base_tasks, seed=202)
}
# Tesztelés miatt kellett csak
# for name, order in experiment_orders.items():
#     print(f"[{name}]: {' -> '.join(order)}")

task_classes = {
    'pathmnist': 9,
    'dermamnist': 7,
    'octmnist': 4,
    'pneumoniamnist': 2,
    'organamnist': 11
}

# Ha már létezik korábbról mentett single-task referencia, betöltjük:
single_ref_path = "results/single_task_reference.json"
b_single_dict = None
if os.path.exists(single_ref_path):
    with open(single_ref_path, "r") as f:
        b_single_dict = json.load(f)

order_summary_rows = []
CURRENT_SEED = set_seed(42)
epochs = 1 # Gyors tesztfutáshoz 1, élesben növelhető

for exp_name, current_order in experiment_orders.items():
    print(f"\n==================================================")
    print(f" FUTTATÁS: {exp_name.upper()}")
    print(f" Sorrend: {' -> '.join(current_order)}")
    print(f"==================================================")

    # Friss modell példányosítása az adott sorrendhez
    model = MultiHeadResNet18(task_classes).to(device)
    T = len(current_order)
    R_matrix = np.zeros((T, T))

    for i, train_task in enumerate(current_order):
        train_loader, val_loader, _, info = get_dataloaders(train_task, batch_size=128)
        task_type = info['task']
        criterion = nn.BCEWithLogitsLoss() if task_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        # Tanítás
        for epoch in range(1, epochs + 1):
            model.train()
            for images, labels in train_loader:
                images = images.to(device)
                optimizer.zero_grad()
                outputs = model(images, task_name=train_task)
                labels = labels.to(device).float() if task_type == 'multi-label, binary-class' else labels.squeeze(-1).long().to(device)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

        # Kiértékelés az ÖSSZES feladaton a sorrend szerint (TxT mátrix sorának kitöltése)
        model.eval()
        with torch.no_grad():
            for j, eval_task in enumerate(current_order):
                _, _, eval_test_loader, eval_info = get_dataloaders(eval_task, batch_size=128)
                eval_type = eval_info['task']
                eval_crit = nn.BCEWithLogitsLoss() if eval_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()
                test_metrics = evaluate_detailed(model, eval_test_loader, eval_type, eval_crit, eval_task=eval_task)
                R_matrix[i, j] = test_metrics["accuracy"]

    # 1. Hőtérkép mentése
    save_publication_matrix(
        R_matrix=R_matrix,
        task_order=current_order,
        experiment_name=f"{exp_name}_seed{CURRENT_SEED}"
    )

    # 2. Metrikák számítása és mentése
    metrics = compute_and_save_cl_metrics(
        R_matrix=R_matrix,
        task_order=current_order,
        seed=CURRENT_SEED,
        experiment_name=exp_name,
        b_single_dict=b_single_dict
    )

    # 3. Összefoglaló sor készítése
    order_summary_rows.append({
        "Sorrend típus": exp_name,
        "Végső Átl. Pontosság (A_T) %": metrics["final_average_accuracy"],
        "Visszamenőleges Transzfer (BWT) %": metrics["backward_transfer"],
        "Átlagos Felejtés (F) %": metrics["average_forgetting"],
        "FWT (Single-ref) %": metrics["forward_transfer_single"]
    })

# Végső összehasonlító táblázat kiírása és mentése
df_order_comparison = pd.DataFrame(order_summary_rows)
df_order_comparison.to_csv("results/task_order_comparison.csv", index=False)

print("\n==================================================")
print(" 2. MUNKACSOMAG - TASK ORDER ÖSSZEHASONLÍTÁS:")
print("==================================================")
display(df_order_comparison)


 FUTTATÁS: SIMILAR_FIRST
 Sorrend: dermamnist -> pathmnist -> organamnist -> pneumoniamnist -> octmnist
Ábra sikeresen elmentve: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\similar_first_seed42_heatmap.png
Eredmények elmentve:
  -> JSON: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\similar_first_seed42_metrics.json

 FUTTATÁS: DISSIMILAR_ALTERNATING
 Sorrend: dermamnist -> octmnist -> pathmnist -> pneumoniamnist -> organamnist
Ábra sikeresen elmentve: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\dissimilar_alternating_seed42_heatmap.png
Eredmények elmentve:
  -> JSON: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\dissimilar_alternating_seed42_metrics.json

 FUTTATÁS: RANDOM_ORDER_1
 Sorrend: organamnist -> dermamnist -> octmnist -> pneumoniamnist -> pathmnist
Ábra sikeresen elmentve: C:\Munka\1. Féléves tárgyak\Fejlett gépi tanuló modellek\Projekt\results\random_order_1_seed

,Sorrend típus,Végső Átl. Pontosság (A_T) %,Visszamenőleges Transzfer (BWT) %,Átlagos Felejtés (F) %,FWT (Single-ref) %
0,similar_first,39.6452,-39.7385,39.7385,2.0194
1,dissimilar_alternating,49.8110,-32.8069,32.8069,7.8732
2,random_order_1,48.6148,-32.6067,32.6067,5.5359
3,random_order_2,43.2543,-38.4424,38.4424,4.4729
